# Mel Frequency Cepstrum Coefficients

In der heutigen Übung betrachten wir Mel Frequency Cepstrum Coefficients, die Standardmerkmale der Sprachverarbeitung. 

Wir führen zuerst ein k-Means-Clustern von Vokalen durch,
anschließend implementieren wir eine Geschlechtserkennung mit KNN.

Sie benoetigen zusaetzlich das Paket *soundfile* um Audiodateien zu lesen und 
*python_speech_features* zur MFCC-Berechnung.

# MFCC Berechnung

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import soundfile as sf
import python_speech_features as mfcc

Folgende Methode berechnet MFCCs auf einer Audiodatei mit Hilfe des Paketes python_speech_features und speichert das Ergebnis 

In [2]:
def extract_feats(audiof):
    """extract 20 dim mfcc features from an audio"""
    
    audio,samplerate= sf.read(audiof);
    mfcc_feat = mfcc.mfcc(audio, samplerate, 0.025, 0.01 , 20, appendEnergy=False)

    return mfcc_feat

## Geschlechtsklassifikation anhand von Sprachdaten


- Im Repository finden Sie Sprachdaten aus dem LibriSpeech-Corpus.
- Es existieren zwei Verzeichnisse mit Sprachdaten. Die Verzeichnisse sind in Unterverzeichnisse aufgeteilt. Jedes Unterverzeichnis beinhaltet Sprachdaten eines Sprechers, diese sind wiederum in Sessions unterteilt. In einer Session existieren mehrere Sprachdaten.
    - eines zum Training: *train-clean*
    - eines zum Testen: *test-clean-10spk*
- Es existeren utterance to speaker maps. Die Maps enthalten zwei Spalten. In der ersten Spalte steht die SprecherID, in der zweiten das label ('F' --> female, 'M' --> male) 
    - utt2spk.train
    - utt2spk.test-10spk





### Teil 1

Aufgaben:

1. Schreiben Sie Code, der die Sprachdaten einliest und MFCCs **pro Session eines Sprechers** berechnet. Am Ende erhalten Sie eine MFCC-Matrix pro Session.
2. Berechnen Sie den Mittelwert, die Varianz, sowie Minimum und Maximum der Merkmale einer Session.
3. Trainieren Sie mit diesen einen KNN-Klassifikator zur Geschlechtserkennung auf den *train-clean*-Daten
4. Testen Sie ihr System auf den *test-clean-10spk*-Daten

In [3]:
def collect_data(directory, user_ids):
    vectors = list()
    labels = list()
    os.chdir(directory)
    for i, uid in enumerate(user_ids):
        os.chdir(str(uid))
        for s in os.listdir("."):
            os.chdir(s)
            
            features_session = None 
            
            for entry in os.listdir("."):
                if not entry.endswith(".flac"):
                    continue
                if features_session is None: 
                    features_session = extract_feats(entry).ravel()
                else: 
                    np.concatenate((features_session, extract_feats(entry).ravel()))

            assert features_session.shape[0] % 20 == 0
            x1 = variance_session = np.var(features_session)
            x2 = average_session = np.average(features_session)
            x3 = min_session = np.min(features_session)
            x4 = max_session = np.max(features_session)
            label = train_gender[i] 
            vec = np.array([x1, x2, x3, x4])
            vectors.append(vec)
            labels.append(label)
    
            os.chdir("..")
                    
            
        os.chdir("..") # str(uid)
    os.chdir("..") # <directory>

    return vectors, labels

In [4]:
import matplotlib.pyplot as plt
import numpy as np
import os
import soundfile as sf
import python_speech_features as mfcc

# read data4
os.chdir("data")

train_gender = dict() 
with open('utt2spk.train', 'r') as file:
    for line in file:
        line = line.rstrip()
        parts = line.split(" ")
        assert (parts[-1] in ['M', 'F'])
        
        train_gender[int(parts[0])] = parts[-1]
    
test_gender = dict() 
with open('utt2spk.test-10spk', 'r') as file:
    for line in file:
        line = line.rstrip()
        parts = line.split(" ")
        assert (parts[-1] in ['M', 'F'])
        test_gender[int(parts[0])] = parts[-1]

def label_is(s: str):
    return 0 if s == 'M' else 1

def get_char(i) -> str: 
    return 'M' if i == 0 else 'F'

train_uids = sorted(train_gender.keys())
train_gender = [label_is(train_gender[k]) for k in train_uids ] 

test_uids = sorted(test_gender.keys()) 
test_gender = [label_is(test_gender[k]) for k in test_uids ] 

train_vectors, train_labels = collect_data("train-clean", train_uids)
test_vectors, test_labels = collect_data("test-clean-10spk", test_uids)
os.chdir("..") # "data" 

In [5]:
assert len(train_labels) == len(train_vectors)

In [6]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=11)

knn.fit(train_vectors, train_labels)

predicted = knn.predict(test_vectors)
print(predicted)
print(np.array(test_labels))

[1 1 1 1 1 1 0 0 1 1 1 1 1 1 1 1 1 1 0 0]
[0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 1 1 1]


### Teil 2

Anstatt für jede Session *Funktionale*, also Mittelwert-, Varianz-, Min-, Maxvektor auf den MFCCs zu berechnen und ggf. zu konkatenieren, können MFCCs mit dem KNN auch frame-weise klassifiziert werden. Dann erzeugt jeder MFCC-Vektor ein Klassifikationsergebnis. Anschließend müssen die frame-weisen Entscheidungen gemittelt werden um pro Session ein finales Ergebnis zu erhalten. 

Implementieren Sie dieses Vorgehen und vergleichen Sie die Ergebnisse mit denen aus Teil 1.